In [1]:
import pandas as pd
import numpy as np

In [2]:
goal_scorers = pd.read_csv(r"..\data\goalscorers_processed.csv")
results = pd.read_csv(r"..\data\results_updated.csv")
shootouts = pd.read_csv(r"..\data\shootouts.csv")
former_names = pd.read_csv(r"..\data\former_names.csv")

In [3]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,0


In [4]:
former_names.head()

,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01


In [5]:
results['date'] = pd.to_datetime(results['date'])
former_names['start_date'] = pd.to_datetime(former_names['start_date'])
former_names['end_date'] = pd.to_datetime(former_names['end_date']).fillna(pd.to_datetime('2030-01-01'))

def get_current_name(team_name, match_date):
    # Finding rows where team matches former name and falls within the historical window
    mask = (former_names['former'] == team_name) & \
           (former_names['start_date'] <= match_date) & \
           (former_names['end_date'] >= match_date)
    
    matching_rows = former_names[mask]

    if not matching_rows.empty:
        return matching_rows.iloc[0]['current']
    return team_name

# apply current name to all team names
results['home_team'] = results.apply(lambda row: get_current_name(row['home_team'], row['date']), axis=1)
results['away_team'] = results.apply(lambda row: get_current_name(row['away_team'], row['date']), axis=1)

In [6]:
results['tournament'].unique()

array(['Friendly', 'British Home Championship', 'Évence Coppée Trophy',
       'Muratti Vase', 'Copa Lipton', 'Copa Newton',
       'Copa Premio Honor Argentino', 'Olympic Games',
       'Copa Premio Honor Uruguayo', 'Far Eastern Championship Games',
       'Copa Roca', 'Copa América', 'Inter-Allied Games', 'Peace Cup',
       'Open International Championship', 'Soccer Ashes',
       'Copa Chevallier Boutell', 'Nordic Championship',
       'Central European International Cup', 'Baltic Cup', 'Balkan Cup',
       'Central American and Caribbean Games', 'FIFA World Cup',
       'Copa Rio Branco', 'FIFA World Cup qualification',
       'Bolivarian Games', 'CCCF Championship', 'NAFC Championship',
       'Copa Oswaldo Cruz', 'Asian Games', 'Pan American Championship',
       'Copa del Pacífico', "Copa Bernardo O'Higgins",
       'AFC Asian Cup qualification', 'Atlantic Cup', 'AFC Asian Cup',
       'African Cup of Nations', 'Copa Paz del Chaco',
       'Merdeka Tournament', 'UEFA Euro quali

### Tournament weight system

In [7]:
def assign_tournament_weight(tournament):
    # Standardize string, remove accents, and strip whitespace
    import unicodedata
    t = unicodedata.normalize('NFKD', tournament).encode('ASCII', 'ignore').decode('utf-8').lower().strip()

    # Catch all qualifiers early to prevent leakage into main tournament blocks
    if 'qualification' in t or 'qualifying' in t:
        if 'fifa world cup' in t:
            return 3.0
        if any(x in t for x in ['uefa euro', 'copa america']):
            return 2.0
        if any(x in t for x in ['african cup of nations', 'afc asian cup', 'gold cup', 'concacaf nations league']):
            return 1.5
        return 1.0  # Minor regional qualifiers

    # Tier 1: The Global Elite
    if any(x in t for x in ['fifa world cup', 'confederations cup', 'mundialito']):
        return 4.0
    
    # Tier 2: Top-Tier Continental Elite
    if any(x in t for x in ['uefa euro', 'copa america', 'conmebol-uefa']):
        return 3.0
    
    # Tier 3: Secondary Continental Giants & Olympics
    if any(x in t for x in ['african cup of nations', 'afc asian cup', 'gold cup', 'concacaf championship', 'oceania nations cup', 'olympic games']):
        return 2.5
    
    # Tier 4: Major Continental Leagues & Prestige Regional Cups
    if any(x in t for x in ['uefa nations league', 'concacaf nations league', 'arab cup', 'gulf cup', 'cafa nations cup', 'aff championship', 'asean championship', 'eaff championship', 'dynasty cup']):
        return 2.0
    
    # Tier 5: Minor regional leagues and other tournaments 
    if 'friendly' not in t and not any(x in t for x in ['series', 'challenge cup', 'anniversary']):
        return 1.5
        
    # 7. Tier 6: Friendlies and minor invitational tournament
    return 1.0


In [8]:
results['tournament_weight'] = results['tournament'].apply(assign_tournament_weight)
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight
2405,1938-10-23,Poland,Norway,2.0,2.0,Friendly,Warsaw,Poland,0,1.0
17861,1991-07-06,Malawi,Kenya,0.0,2.0,Friendly,Blantyre,Malawi,0,1.0
6382,1965-08-02,China,Indonesia,3.0,0.0,Friendly,Pyongyang,North Korea,1,1.0
16521,1989-01-22,Liberia,Kenya,0.0,0.0,FIFA World Cup qualification,Monrovia,Liberia,0,3.0
9523,1973-10-10,Germany,Austria,4.0,0.0,Friendly,Hanover,Germany,0,1.0


### ELO System

In [9]:
results = results.sort_values(by='date').reset_index(drop=True)
results['date'] = pd.to_datetime(results['date'])

# Baseline elo of 1500
elo_ratings = {}
def get_elo(team):
    return elo_ratings.setdefault(team, 1500)

# Elo of teams before the match happens
home_elos = []
away_elos = []

# Initialized beginning year, but will update to row's previous year for elo decay
previous_year = results.loc[0, 'date'].year

# Calculate and update elos for every match
for i, row in results.iterrows():
    home, away = row['home_team'], row['away_team']
    current_year = row['date'].year

    if current_year != previous_year:
        for team in elo_ratings:
            elo_ratings[team] = elo_ratings[team] + 0.1 * (1500 - elo_ratings[team]) # reduction of 10% each year due to outside changes (new season, potential squad and coaching changes)
        
        # updating prevous_year so decay only applies once per year
        previous_year = current_year

    # get elo before match starts
    h_elo = get_elo(home)
    a_elo = get_elo(away)

    home_elos.append(h_elo)
    away_elos.append(a_elo)

    # Expected scores using standard Elo formula
    #       E_A = 1 / (1 + 10^((R_B - R_A) / 400))
    exp_home = 1 / (1 + 10 ** ((a_elo - h_elo) / 400))
    exp_away = 1 - exp_home

    # Outcome weights (Win = 1, Draw = 0.5, Loss = 0)
    if row['home_score'] > row['away_score']:
        actual_home, actual_away = 1.0, 0.0
    elif row['home_score'] < row['away_score']:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # K-Factor to define how volatile rating shifts are, scaled by the tournament weight
    K = 30 * row['tournament_weight']

    # Update ratings
    elo_ratings[home] = h_elo + K * (actual_home - exp_home)
    elo_ratings[away] = a_elo + K * (actual_away - exp_away)

results['home_elo'] = home_elos
results['away_elo'] = away_elos

# ELO-based win probability
results['home_win_prob'] = 1 / (1 + 10 ** ((results['away_elo'] - results['home_elo']) / 400))
results['away_win_prob'] = 1 - results['home_win_prob']

In [10]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,home_elo,away_elo,home_win_prob,away_win_prob
44189,2021-06-26,Wales,Denmark,0.0,4.0,UEFA Euro,Amsterdam,Netherlands,1,3.0,1736.416233,1756.484535,0.471151,0.528849
26968,2003-02-12,Georgia,Moldova,2.0,2.0,Friendly,Tbilisi,Georgia,0,1.0,1507.481228,1323.141801,0.742909,0.257091
39596,2016-03-29,Mexico,Canada,2.0,0.0,FIFA World Cup qualification,Mexico City,Mexico,0,3.0,1801.546793,1556.090073,0.804233,0.195767
30618,2007-01-21,Saudi Arabia,Qatar,1.0,1.0,Gulf Cup,Abu Dhabi,United Arab Emirates,1,2.0,1660.684310,1608.270895,0.574862,0.425138
29288,2005-06-18,Gabon,Rwanda,3.0,0.0,FIFA World Cup qualification,Libreville,Gabon,0,3.0,1488.718365,1459.471390,0.541991,0.458009


### 5-match history

In [11]:
# result from home team's perspective
results['result'] = np.where(results['home_score'] > results['away_score'], 'W',
                             np.where(results['home_score'] < results['away_score'], 'L', 'D'))

# Rolling win rate
team_matches = pd.concat([
    results[['date', 'home_team', 'result']].rename(columns={'home_team': 'team', 'result': 'res'}),
    results[['date', 'away_team', 'result']].rename(columns={'away_team': 'team', 'result': 'res'})
]).sort_values('date')

# Map results from team's perspective
team_matches['is_win'] = np.where(
    ((team_matches['team'] == results.loc[team_matches.index, 'home_team']) & (team_matches['res'] == 'W')) |
    ((team_matches['team'] == results.loc[team_matches.index, 'away_team']) & (team_matches['res'] == 'L')), 
    1, 0
)

# Calculating rolling 5-match win rate, shifting by 1 to not include current match result
team_matches['rolling_win_rate'] = team_matches.groupby('team')['is_win'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).fillna(0.33) # filling historical beginnings with even 1/3 split

# Mapping rolling win rates back to main results dataframe
results = results.merge(team_matches[['date', 'team', 'rolling_win_rate']].rename(columns={'rolling_win_rate': 'home_5_game_win_rate'}),
                        left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').drop(columns='team')

results = results.merge(team_matches[['date', 'team', 'rolling_win_rate']].rename(columns={'rolling_win_rate': 'away_5_game_win_rate'}),
                        left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').drop(columns='team')

In [12]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,home_elo,away_elo,home_win_prob,away_win_prob,result,home_5_game_win_rate,away_5_game_win_rate
1106,1925-11-29,Argentina,Paraguay,2.0,0.0,Copa América,Buenos Aires,Argentina,0,3.0,1598.036151,1572.192396,0.537124,0.462876,W,0.2,0.6
25476,2001-01-27,Botswana,Eswatini,1.0,2.0,Friendly,Gaborone,Botswana,0,1.0,1360.448211,1398.390853,0.445612,0.554388,L,0.4,0.6
9770,1973-10-29,Curaçao,Suriname,2.0,2.0,Friendly,Willemstad,Curaçao,0,1.0,1472.297863,1578.689480,0.351503,0.648497,D,0.2,0.4
16063,1987-08-01,Kenya,Tunisia,1.0,0.0,All-African Games,Nairobi,Kenya,0,1.5,1475.844804,1522.571345,0.433158,0.566842,W,0.2,0.0
8282,1970-06-26,Hong Kong,Malaysia,1.0,1.0,Indonesia Tournament,Jakarta,Indonesia,1,1.5,1367.209167,1542.595078,0.267058,0.732942,D,0.2,0.8


In [13]:
# Difference in elo from home and away teams (positive means home team is favored, negative means away team is favored)
results['home_team_favored'] = results['home_elo'] - results['away_elo']

### 5-match rolling goals scored/conceded

In [14]:
# Add goals to the team_matches
team_goals = pd.concat([
    results[['date', 'home_team', 'home_score', 'away_score']].rename(columns={'home_team': 'team', 'home_score': 'goals_scored', 'away_score': 'goals_conceded'}),
    results[['date', 'away_team', 'away_score', 'home_score']].rename(columns={'away_team': 'team', 'away_score': 'goals_scored', 'home_score': 'goals_conceded'})
]).sort_values('date')

# Calculating 5-match rolling averages (shifted by 1 to avoid current game)
team_goals['rolling_goals_scored'] = team_goals.groupby('team')['goals_scored'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(1.2)
team_goals['rolling_goals_conceded'] = team_goals.groupby('team')['goals_conceded'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(1.2)

# Merge back to main results
results = results.merge(team_goals[['date', 'team', 'rolling_goals_scored', 'rolling_goals_conceded']].rename(columns={'rolling_goals_scored': 'home_5_game_goals', 'rolling_goals_conceded': 'home_5_game_conceded'}), left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').drop(columns='team')
results = results.merge(team_goals[['date', 'team', 'rolling_goals_scored', 'rolling_goals_conceded']].rename(columns={'rolling_goals_scored': 'away_5_game_goals', 'rolling_goals_conceded': 'away_5_game_conceded'}), left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').drop(columns='team')

In [15]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_win_prob,away_win_prob,result,home_5_game_win_rate,away_5_game_win_rate,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded
2021,1921-05-30,Japan,Philippines,0.0,3.0,Friendly,Shanghai,China,1,1.0,...,0.458231,0.541769,L,0.0,0.4,-29.092105,1.25,9.5,3.0,0.4
9279,1956-11-14,Argentina,Uruguay,2.0,2.0,Friendly,Buenos Aires,Argentina,0,1.0,...,0.620348,0.379652,D,0.8,0.2,85.299853,1.20,0.4,1.2,1.8
58429,2021-06-04,North Macedonia,Kazakhstan,4.0,0.0,Friendly,Skopje,North Macedonia,0,1.0,...,0.796368,0.203632,W,0.4,0.0,236.907072,2.00,1.2,0.6,1.6
12020,1966-08-20,Singapore,Taiwan,4.0,1.0,Merdeka Tournament,Ipoh,Malaysia,1,1.5,...,0.392754,0.607246,W,0.2,0.2,-75.697560,3.60,1.2,1.0,4.0
30072,1989-06-18,Aruba,Saint Kitts and Nevis,1.0,4.0,CFU Caribbean Cup qualification,Oranjestad,Aruba,0,1.0,...,0.453858,0.546142,L,0.0,0.6,-32.154113,0.00,4.6,1.0,1.0


### Historical Head-to-Head Advantage or Disadvantage

In [16]:
def calculate_h2h(df):
    # Sort names by home_team, away_team to compared historical matchups against the same teams
    teams = np.sort(df[['home_team', 'away_team']].values, axis=1)
    df['matchup_key'] = teams[:, 0] + "_" + teams[:, 1]

    # Tracking wins for team that appears first in the key
    df['team_a_won'] = np.where(
        ((df['home_team'] == df['matchup_key'].str.split('_').str[0]) & (df['result'] == 'W')) |
        ((df['away_team'] == df['matchup_key'].str.split('_').str[0]) & (df['result'] == 'L')), 
        1, 0
    )

    # Calculating win rate of Team A over Team B
    df['h2h_team_a_win_rate'] = df.groupby('matchup_key')['team_a_won'].transform(lambda x: x.shift(1).expanding().mean().fillna(0.5))

    # Translate this back to the perspective of the current home_team
    df['home_historical_advantage'] = np.where(df['home_team'] == df['matchup_key'].str.split('_').str[0], 
                                        df['h2h_team_a_win_rate'], 
                                        1 - df['h2h_team_a_win_rate'])
    
    return df.drop(columns=['matchup_key', 'team_a_won', 'h2h_team_a_win_rate'])

results = calculate_h2h(results)

In [17]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,away_win_prob,result,home_5_game_win_rate,away_5_game_win_rate,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded,home_historical_advantage
39643,2001-03-11,Palestine,Malaysia,1.0,0.0,FIFA World Cup qualification,So Kon Po,Hong Kong,1,3.0,...,0.688445,W,0.0,0.4,-137.733892,1.0,1.6,1.2,2.4,0.500000
14090,1969-06-29,Zambia,Cameroon,2.0,2.0,African Cup of Nations qualification,Ndola,Zambia,0,1.5,...,0.371616,D,0.4,0.2,91.252358,1.8,2.0,2.0,1.4,1.000000
58425,2021-06-04,Latvia,Lithuania,3.0,1.0,Baltic Cup,Riga,Latvia,0,1.5,...,0.502315,W,0.2,0.2,-1.608513,2.0,1.6,0.4,1.8,0.491525
38450,2000-02-10,Hong Kong,Singapore,0.0,0.0,Friendly,Victoria,Hong Kong,0,1.0,...,0.621694,D,0.6,0.4,-86.293095,2.2,1.6,1.0,0.8,0.352941
55765,2018-03-22,Dominican Republic,Turks and Caicos Islands,4.0,0.0,Friendly,Santo Domingo,Dominican Republic,0,1.0,...,0.379069,W,0.6,0.2,85.730133,2.0,1.2,1.2,3.8,0.500000


### 5-game Goal Differential

In [18]:
results['home_5_game_goal_differential'] = results['home_5_game_goals'] - results['home_5_game_conceded']
results['away_5_game_goal_differential'] = results['away_5_game_goals'] - results['away_5_game_conceded']

In [19]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_5_game_win_rate,away_5_game_win_rate,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded,home_historical_advantage,home_5_game_goal_differential,away_5_game_goal_differential
42703,2004-06-20,Bermuda,El Salvador,2.0,2.0,FIFA World Cup qualification,Hamilton,Bermuda,0,3.0,...,0.2,0.2,-112.484619,0.8,2.2,1.2,2.0,0.333333,-1.4,-0.8
53176,2015-06-10,Saint Vincent and the Grenadines,Guyana,2.0,2.0,FIFA World Cup qualification,Kingstown,Saint Vincent and the Grenadines,0,3.0,...,0.8,0.4,-6.517657,2.2,0.6,1.2,1.2,0.500000,1.6,0.0
53417,2015-08-29,Laos,Cambodia,2.0,1.0,Friendly,Bangkok,Thailand,1,1.0,...,0.0,0.4,-10.182783,0.8,2.4,0.8,1.2,0.228571,-1.6,-0.4
52721,2014-11-14,Hungary,Finland,1.0,0.0,UEFA Euro qualification,Budapest,Hungary,0,2.0,...,0.6,0.4,-8.611215,1.4,0.6,1.2,1.0,0.857143,0.8,0.2
2878,1924-05-25,Uruguay,Argentina,2.0,0.0,Copa Lipton,Montevideo,Uruguay,0,1.5,...,0.6,0.4,34.182479,1.6,0.6,2.4,0.8,0.047480,1.0,1.6


In [20]:
results.to_csv(r"..\data\results_engineered.csv", index=False)

# Shootouts

In [21]:
shootouts.head()

,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


In [22]:
shootouts['date'] = pd.to_datetime(shootouts['date'])
shootouts['home_team'] = shootouts.apply(lambda row: get_current_name(row['home_team'], row['date']), axis=1)
shootouts['away_team'] = shootouts.apply(lambda row: get_current_name(row['away_team'], row['date']), axis=1)
shootouts['winner'] = shootouts.apply(lambda row: get_current_name(row['winner'], row['date']), axis=1)

# shootouts by date and matchup
team_shootouts = pd.concat([
    shootouts[['date', 'home_team', 'winner']].rename(columns={'home_team': 'team'}),
    shootouts[['date', 'away_team', 'winner']].rename(columns={'away_team': 'team'})
]).sort_values('date')

# check if team won the shootout
team_shootouts['is_shootout_win'] = (team_shootouts['team'] == team_shootouts['winner']).astype(int)

# historical matchup penalty win rate
team_shootouts['historical_matchup_penalty_win_rate'] = (
    team_shootouts.groupby('team')['is_shootout_win']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.50) # If a team has never been in a shootout, default to a 50/50 baseline
)

team_shootouts = team_shootouts.drop_duplicates(subset=['date', 'team'], keep='last')

# mapping (date, team) -> win rate
shootout_map = dict(zip(zip(team_shootouts['date'], team_shootouts['team']), team_shootouts['historical_matchup_penalty_win_rate']))

# Use a zip layout to map the keys directly into new columns
results['home_historical_matchup_pen_win_rate'] = pd.Series(zip(results['date'], results['home_team'])).map(shootout_map)
results['away_historical_matchup_pen_win_rate'] = pd.Series(zip(results['date'], results['away_team'])).map(shootout_map)

# fill teams that have zero shootout history with the 50/50 baseline
results['home_historical_matchup_pen_win_rate'] = results['home_historical_matchup_pen_win_rate'].fillna(0.50)
results['away_historical_matchup_pen_win_rate'] = results['away_historical_matchup_pen_win_rate'].fillna(0.50)

In [23]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded,home_historical_advantage,home_5_game_goal_differential,away_5_game_goal_differential,home_historical_matchup_pen_win_rate,away_historical_matchup_pen_win_rate
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,0,1.0,...,0.000000,1.200000,1.200000,1.200000,1.200000,0.500000,0.000000,0.000000,0.5,0.5
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,0,1.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5,0.5
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,0,1.0,...,-27.000000,1.000000,2.000000,2.000000,1.000000,0.500000,-1.000000,1.000000,0.5,0.5
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,0,1.0,...,-4.794017,1.666667,1.333333,1.333333,1.666667,0.333333,0.333333,-0.333333,0.5,0.5
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,0,1.0,...,3.942085,1.500000,1.750000,1.750000,1.500000,0.750000,-0.250000,0.250000,0.5,0.5


### Psychological impact on performance based on order of penalties taken

In [24]:
shootouts['date'] = pd.to_datetime(shootouts['date'])

# Only apply name mapping where first_shooter is NOT null
not_null_mask = shootouts['first_shooter'].notna()
shootouts.loc[not_null_mask, 'first_shooter'] = shootouts[not_null_mask].apply(
    lambda row: get_current_name(row['first_shooter'], row['date']), axis=1
)

# If first_shooter is NaN, second_shooter becomes NaN
shootouts['second_shooter'] = np.where(
    shootouts['first_shooter'].isna(), 
    np.nan, 
    np.where(shootouts['first_shooter'] == shootouts['home_team'], shootouts['away_team'], shootouts['home_team'])
)

# Drop rows where no shootout happened before stacking historical records
valid_shootouts = shootouts.dropna(subset=['first_shooter', 'second_shooter'])

# Tracking performance when shooting first
first_shooters = valid_shootouts[['date', 'first_shooter', 'winner']].rename(columns={'first_shooter': 'team'})
first_shooters['won_as_first'] = (first_shooters['team'] == first_shooters['winner']).astype(int)
first_shooters['historical_first_kick_clutch'] = (
    first_shooters.groupby('team')['won_as_first']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.60)
)

# Track performance when shooting second
second_shooters = valid_shootouts[['date', 'second_shooter', 'winner']].rename(columns={'second_shooter': 'team'})
second_shooters['won_as_second'] = (second_shooters['team'] == second_shooters['winner']).astype(int)
second_shooters['historical_second_kick_resilience'] = (
    second_shooters.groupby('team')['won_as_second']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.40)
)

# Drop duplicates to create clean daily lookups
first_lookup = first_shooters.drop_duplicates(subset=['date', 'team'], keep='last')
second_lookup = second_shooters.drop_duplicates(subset=['date', 'team'], keep='last')

# Creating maps for first and second penalty shooters
first_map = dict(zip(zip(first_lookup['date'], first_lookup['team']), first_lookup['historical_first_kick_clutch']))
second_map = dict(zip(zip(second_lookup['date'], second_lookup['team']), second_lookup['historical_second_kick_resilience']))

# Map to main results dataset
results['home_pen_win_rate_shot_first'] = pd.Series(zip(results['date'], results['home_team'])).map(first_map).fillna(0.60)
results['away_pen_win_rate_shot_first'] = pd.Series(zip(results['date'], results['away_team'])).map(first_map).fillna(0.60)

results['home_pen_win_rate_shot_second'] = pd.Series(zip(results['date'], results['home_team'])).map(second_map).fillna(0.40)
results['away_pen_win_rate_shot_second'] = pd.Series(zip(results['date'], results['away_team'])).map(second_map).fillna(0.40)

In [25]:
shootouts.head()

,date,home_team,away_team,winner,first_shooter,second_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN,NaN


In [26]:
shootouts.sample(5)

,date,home_team,away_team,winner,first_shooter,second_shooter
169,1993-01-23,Japan,Switzerland,Switzerland,NaN,NaN
359,2005-04-16,Namibia,Botswana,Botswana,NaN,NaN
499,2016-11-12,Nepal,Laos,Nepal,NaN,NaN
8,1973-07-26,Malaysia,Kuwait,Malaysia,NaN,NaN
255,1999-07-31,Namibia,South Africa,Namibia,NaN,NaN


In [27]:
results.sample(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,away_5_game_conceded,home_historical_advantage,home_5_game_goal_differential,away_5_game_goal_differential,home_historical_matchup_pen_win_rate,away_historical_matchup_pen_win_rate,home_pen_win_rate_shot_first,away_pen_win_rate_shot_first,home_pen_win_rate_shot_second,away_pen_win_rate_shot_second
13644,1969-02-02,Cameroon,Uganda,2.0,0.0,African Cup of Nations qualification,Yaoundé,Cameroon,0,1.5,...,2.0,0.397647,0.200000,0.4,0.5,0.5,0.6,0.6,0.4,0.4
44585,2006-09-03,Guinea,Algeria,0.0,0.0,African Cup of Nations qualification,Conakry,Guinea,0,1.5,...,1.4,0.500000,1.000000,-0.8,0.5,0.5,0.6,0.6,0.4,0.4
62199,2024-10-11,South Africa,Congo,5.0,0.0,African Cup of Nations qualification,Gqeberha,South Africa,0,1.5,...,2.6,0.900000,0.600000,-1.8,0.5,0.5,0.6,0.6,0.4,0.4
23789,1979-09-01,Tonga,Tuvalu,3.0,5.0,South Pacific Games,Nausori,Fiji,1,1.5,...,18.0,0.500000,-8.000000,-18.0,0.5,0.5,0.6,0.6,0.4,0.4
26154,1983-06-05,Iceland,Malta,1.0,0.0,UEFA Euro qualification,Reykjavík,Iceland,0,2.0,...,2.6,0.000000,-1.000000,-1.8,0.5,0.5,0.6,0.6,0.4,0.4
7957,1949-06-02,Sweden,Republic of Ireland,3.0,1.0,FIFA World Cup qualification,Solna,Sweden,0,3.0,...,1.4,0.500000,0.400000,-1.0,0.5,0.5,0.6,0.6,0.4,0.4
63758,2026-05-22,Mexico,Ghana,2.0,0.0,Friendly,Puebla,Mexico,0,1.0,...,2.0,1.000000,1.200000,-1.4,0.5,0.5,0.6,0.6,0.4,0.4
35893,1997-07-03,Frøya,Hitra,4.0,1.0,Island Games,Grouville,Jersey,1,1.5,...,2.4,1.000000,-2.666667,-1.4,0.5,0.5,0.6,0.6,0.4,0.4
17116,1973-04-08,Ethiopia,Tanzania,2.0,1.0,African Cup of Nations qualification,Addis Ababa,Ethiopia,0,1.5,...,0.6,0.445161,0.600000,1.4,0.5,0.5,0.6,0.6,0.4,0.4
56263,2018-09-09,France,Netherlands,2.0,1.0,UEFA Nations League,Paris,France,0,2.0,...,0.8,0.461538,1.200000,0.6,0.5,0.5,0.6,0.6,0.4,0.4


In [28]:
results['home_pen_win_rate_shot_first'].unique()

array([0.6       , 1.        , 0.        , 0.5       , 0.33333333,
       0.66666667, 0.25      , 0.75      , 0.8       , 0.57142857,
       0.625     , 0.71428571, 0.375     , 0.4       ])

In [29]:
results['away_pen_win_rate_shot_first'].unique()

array([0.6       , 1.        , 0.        , 0.75      , 0.33333333,
       0.5       , 0.66666667, 0.4       , 0.8       , 0.42857143,
       0.25      , 0.55555556, 0.44444444])

In [30]:
results['home_pen_win_rate_shot_second'].unique()

array([0.4       , 0.        , 0.5       , 0.33333333, 1.        ,
       0.25      , 0.2       , 0.71428571, 0.66666667, 0.6       ,
       0.75      , 0.42857143])

In [31]:
results['away_pen_win_rate_shot_second'].unique()

array([0.4       , 1.        , 0.        , 0.33333333, 0.5       ,
       0.25      , 0.66666667, 0.75      , 0.8       , 0.6       ,
       0.63636364, 0.42857143, 0.2       , 0.375     ])

In [32]:
results.to_csv(r"..\data\results_engineered.csv", index=False)

In [33]:
results = pd.read_csv(r"..\data\results_engineered.csv")

In [34]:
goal_scorers.head()

,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,0,0
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,0,0
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,0,0
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,0,0
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,0,0


In [35]:
goal_scorers.sample(5)

,date,home_team,away_team,team,scorer,minute,own_goal,penalty
2505,1956-12-05,England,Denmark,England,Tommy Taylor,22.0,0,0
32131,2012-09-11,Bosnia and Herzegovina,Latvia,Bosnia and Herzegovina,Zvjezdan Misimović,12.0,0,0
29990,2010-06-28,Brazil,Chile,Brazil,Juan,34.0,0,0
38055,2019-01-07,Iran,Yemen,Iran,Sardar Azmoun,53.0,0,0
47292,2025-11-15,Georgia,Spain,Spain,Ferran Torres,35.0,0,0


In [36]:
# percentage of goals in clutch time 
len(goal_scorers[goal_scorers['minute'] >= 75]) / len(goal_scorers)

0.23267564463931517

In [37]:
# percentage of own goals
len(goal_scorers[goal_scorers['own_goal'] == 1]) / len(goal_scorers)

0.019605386783589102

In [38]:
results['date'] = pd.to_datetime(results['date'])
results = results.sort_values('date').reset_index(drop=True)

goal_scorers['date'] = pd.to_datetime(goal_scorers['date'])
goal_scorers['team'] = goal_scorers.apply(lambda row: get_current_name(row['team'], row['date']), axis=1)
goal_scorers = goal_scorers.sort_values('date').reset_index(drop=True)

# tracking goals in the clutch time and if a goal scored was an own goal
goal_scorers['is_late_goal'] = (goal_scorers['minute'] >= 75).astype(int)
goal_scorers['is_own_goal'] = (goal_scorers['own_goal'] == True).astype(int)

# historical averages for clutch goals
goal_scorers['hist_clutch_scoring_rate'] = (
    goal_scorers.groupby('team')['is_late_goal'].transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.23) # Global baseline: roughly 23% of goals naturally occur late game
)

goal_scorers['hist_opponent_own_goal_rate'] = (
    goal_scorers.groupby('team')['is_own_goal'].transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.02) # Global baseline: roughly 2% of goals are opponent own goals
)

# Drop duplicate rows for the same date and team, keeping the latest pre-match baseline
lookup_df = goal_scorers[['date', 'team', 'hist_clutch_scoring_rate', 'hist_opponent_own_goal_rate']].drop_duplicates(subset=['date', 'team'], keep='last')
lookup_df = lookup_df.sort_values('date').reset_index(drop=True)

# merge data for home and away teams
results = pd.merge_asof(
    results, 
    lookup_df.rename(columns={'team': 'home_team', 'hist_clutch_scoring_rate': 'home_clutch_scoring_rate', 'hist_opponent_own_goal_rate': 'home_og_inducer_rate'}),
    on='date', 
    by='home_team', 
    direction='backward'
)

# Merge for Away Team
results = pd.merge_asof(
    results, 
    lookup_df.rename(columns={'team': 'away_team', 'hist_clutch_scoring_rate': 'away_clutch_scoring_rate', 'hist_opponent_own_goal_rate': 'away_og_inducer_rate'}),
    on='date', 
    by='away_team', 
    direction='backward'
)

results[['home_clutch_scoring_rate', 'away_clutch_scoring_rate']] = results[['home_clutch_scoring_rate', 'away_clutch_scoring_rate']].fillna(0.23)
results[['home_og_inducer_rate', 'away_og_inducer_rate']] = results[['home_og_inducer_rate', 'away_og_inducer_rate']].fillna(0.02)

In [39]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_historical_matchup_pen_win_rate,away_historical_matchup_pen_win_rate,home_pen_win_rate_shot_first,away_pen_win_rate_shot_first,home_pen_win_rate_shot_second,away_pen_win_rate_shot_second,home_clutch_scoring_rate,home_og_inducer_rate,away_clutch_scoring_rate,away_og_inducer_rate
22672,1977-07-19,Iraq,Libya,0.0,0.0,Merdeka Tournament,Kuala Lumpur,Malaysia,1,1.5,...,0.5,0.5,0.6,0.6,0.4,0.4,0.214286,0.000000,0.000000,0.0
22580,1977-04-27,Germany,Northern Ireland,5.0,0.0,Friendly,Cologne,Germany,0,1.0,...,0.5,0.5,0.6,0.6,0.4,0.4,0.259740,0.017316,0.148649,0.0
44755,2006-10-11,Qatar,Hong Kong,2.0,0.0,AFC Asian Cup qualification,Doha,Qatar,0,1.5,...,0.5,0.5,0.6,0.6,0.4,0.4,0.248062,0.023256,0.253333,0.0
14046,1969-06-29,Cameroon,Zambia,2.0,1.0,African Cup of Nations qualification,Yaoundé,Cameroon,0,1.5,...,0.5,0.5,0.6,0.6,0.4,0.4,0.000000,0.000000,0.200000,0.0
30838,1990-12-11,Central African Republic,Gabon,0.0,1.0,UDEAC Cup,Brazzaville,Congo,1,1.5,...,0.5,0.5,0.6,0.6,0.4,0.4,0.230000,0.020000,0.250000,0.0


In [40]:
# =====================================================
# ===================== TARGET ========================
# =====================================================

# (0 = Away Win, 1 = Draw, 2 = Home Win)
results['match_outcome'] = np.where(results['home_score'] > results['away_score'], 2, np.where(results['home_score'] == results['away_score'], 1, 0))

In [41]:
results.to_csv(r"..\data\results_engineered.csv", index=False)